# Soybean Yield Prediction Models

This notebook trains rolling out-of-sample soybean yield prediction models using the model-ready county-year panel produced by `01_data_preparation.ipynb`.

The main goal is to compare July-end, August-end, and September-end feature windows and test whether prediction accuracy improves as more growing-season information becomes available. The notebook also converts county-level predictions into aggregate satellite-implied yield signals for later WASDE and futures-market analysis.


In [ ]:
# ============================================================
# 0. Setup
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# If running in Google Colab
from google.colab import drive
drive.mount("/content/gdrive")

PROJECT_DIR = Path("/content/gdrive/MyDrive/Soybean_Project_GEE")
PROCESSED_DIR = PROJECT_DIR / "processed"
USDA_DIR = PROJECT_DIR / "USDA_County_Outcomes"
OUTPUT_TABLE_DIR = PROJECT_DIR / "outputs" / "tables"

OUTPUT_TABLE_DIR.mkdir(parents=True, exist_ok=True)


## 1. Load modeling inputs

`model_data` is the merged county-year modeling panel from the data preparation notebook.  
`county_outcomes` is the USDA county outcome history and is used later to obtain prior-year harvested acres for acreage-weighted aggregation.


In [ ]:
# ============================================================
# 1. Load model-ready panel and USDA county outcomes
# ============================================================

model_ready_path = PROCESSED_DIR / "model_ready_soybean_15states_2010_2025.csv"
county_outcomes_path = USDA_DIR / "Soybean_Real_County_Outcomes_15CandidateStates_2010_2025.csv"

if not model_ready_path.exists():
    raise FileNotFoundError(f"Missing model-ready file: {model_ready_path}")

if not county_outcomes_path.exists():
    raise FileNotFoundError(f"Missing USDA outcome file: {county_outcomes_path}")

model_data = pd.read_csv(model_ready_path)
county_outcomes = pd.read_csv(county_outcomes_path)

print("Model-ready rows:", len(model_data))
print("Model-ready columns:", model_data.shape[1])
print("Years:", model_data["year"].min(), "-", model_data["year"].max())
print("States:", sorted(model_data["state_alpha"].dropna().unique()))

print("\nCounty outcome rows:", len(county_outcomes))
print("County outcome years:", county_outcomes["year"].min(), "-", county_outcomes["year"].max())

display(model_data.head())
display(county_outcomes.head())


## 2. Define model features and model candidates

The feature windows mimic information available at different points in the growing season:

- **July-end** uses features available through July.
- **August-end** uses features available through August.
- **September-end** uses features available through September.

The models are evaluated using a rolling out-of-sample design: for each test year, models are trained only on prior years and tested on the held-out year.


In [ ]:
# ============================================================
# 2. Import modeling tools
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor


In [ ]:
# ============================================================
# 3. Define feature windows
# ============================================================

target = "yield_bu_acre"

feature_sets = {
    "July-end": [
        "ndvi_06",
        "rain_mm_06",
        "soil_moisture_l2_06",
        "heat_days_gt35c_06",
        "ndvi_07",
        "evi_07",
        "rain_mm_07",
        "heat_days_gt35c_07",
        "soil_moisture_l2_07",
    ],

    "August-end": [
        "ndvi_06",
        "ndvi_07",
        "evi_07",
        "rain_mm_07",
        "soil_moisture_l2_07",
        "heat_days_gt35c_07",
        "ndvi_08",
        "evi_08",
        "rain_mm_08",
        "heat_days_gt35c_08",
        "soil_moisture_l2_08",
    ],

    "September-end": [
        "ndvi_06",
        "ndvi_07",
        "rain_mm_07",
        "soil_moisture_l2_07",
        "heat_days_gt35c_07",
        "ndvi_08",
        "evi_08",
        "rain_mm_08",
        "heat_days_gt35c_08",
        "soil_moisture_l2_08",
        "ndvi_09",
        "evi_09",
        "rain_mm_09",
        "heat_days_gt35c_09",
    ],
}

In [ ]:
# ============================================================
# 4. Define model candidates
# ============================================================

def build_models():
    """
    Return fresh model objects for each rolling forecast year.
    A new model is fitted separately for each test year.
    """
    return {
        "Linear Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LinearRegression())
        ]),

        "Random Forest": RandomForestRegressor(
            n_estimators=300,
            max_depth=6,
            min_samples_leaf=5,
            max_features=0.7,
            random_state=42,
            n_jobs=-1
        ),

        "XGBoost": XGBRegressor(
            objective="reg:squarederror",
            n_estimators=400,
            max_depth=3,
            learning_rate=0.03,
            min_child_weight=5,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=2.0,
            random_state=42,
            n_jobs=-1,
            verbosity=0
        )
    }

print(build_models())

## 3. Rolling out-of-sample yield prediction

For each feature window and test year, the model is trained on all prior years and evaluated on the current test year. A county historical-mean benchmark is included as the baseline for OOS R² comparison.


In [ ]:
# ============================================================
# 5. Rolling out-of-sample function
# ============================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def run_rolling_oos(model_data, core_features, feature_window, test_years=range(2018, 2026)):
    """
    Train rolling out-of-sample models for a given feature window.
    For each test year t, train on years < t and predict year t.
    """

    required_cols = (
        ["state_alpha", "county_ansi", "year", "harvested_acres", target]
        + core_features
    )

    missing_cols = [
        col for col in required_cols
        if col not in model_data.columns
    ]

    if missing_cols:
        raise ValueError(f"Missing columns for {feature_window}: {missing_cols}")

    model_ready_window = model_data.dropna(
        subset=core_features + [target, "harvested_acres"]
    ).copy()

    model_ready_window["year"] = model_ready_window["year"].astype(int)
    model_ready_window["county_ansi"] = (
        model_ready_window["county_ansi"]
        .astype(str)
        .str.zfill(3)
    )

    all_predictions = []

    for test_year in test_years:

        train = model_ready_window[model_ready_window["year"] < test_year].copy()
        test = model_ready_window[model_ready_window["year"] == test_year].copy()

        if train.empty or test.empty:
            print(f"Skipping {feature_window} {test_year}: insufficient data.")
            continue

        X_train = train[core_features]
        y_train = train[target]

        X_test = test[core_features]

        # Historical mean benchmark
        county_historical_mean = (
            train
            .groupby("county_ansi")[target]
            .mean()
        )

        fallback_mean = y_train.mean()

        benchmark_pred = (
            test["county_ansi"]
            .map(county_historical_mean)
            .fillna(fallback_mean)
            .to_numpy()
        )

        benchmark_output = test[
            ["state_alpha", "county_ansi", "year", target, "harvested_acres"]
        ].copy()

        benchmark_output["model"] = "Historical Mean"
        benchmark_output["prediction"] = benchmark_pred
        benchmark_output["feature_window"] = feature_window

        all_predictions.append(benchmark_output)

        # ML models
        models = build_models()

        for model_name, model in models.items():
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)

            output = test[
                ["state_alpha", "county_ansi", "year", target, "harvested_acres"]
            ].copy()

            output["model"] = model_name
            output["prediction"] = predictions
            output["feature_window"] = feature_window

            all_predictions.append(output)

    predictions_oos = pd.concat(all_predictions, ignore_index=True)

    predictions_oos["error"] = (
        predictions_oos["prediction"] - predictions_oos[target]
    )

    predictions_oos["abs_error"] = predictions_oos["error"].abs()
    predictions_oos["squared_error"] = predictions_oos["error"] ** 2

    # Evaluation table
    benchmark_sse = (
        predictions_oos[
            predictions_oos["model"] == "Historical Mean"
        ]["squared_error"].sum()
    )

    results = []

    for model_name, df_model in predictions_oos.groupby("model"):

        y_true = df_model[target]
        y_pred = df_model["prediction"]
        weights = df_model["harvested_acres"]

        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        pooled_r2 = r2_score(y_true, y_pred)

        model_sse = df_model["squared_error"].sum()

        oos_r2_vs_hist_mean = (
            1 - model_sse / benchmark_sse
            if model_name != "Historical Mean"
            else 0.0
        )

        weighted_mae = np.average(
            np.abs(y_true - y_pred),
            weights=weights
        )

        weighted_rmse = np.sqrt(
            np.average(
                (y_true - y_pred) ** 2,
                weights=weights
            )
        )

        results.append({
            "feature_window": feature_window,
            "model": model_name,
            "mae_bu_acre": mae,
            "rmse_bu_acre": rmse,
            "pooled_r2": pooled_r2,
            "oos_r2_vs_hist_mean": oos_r2_vs_hist_mean,
            "acreage_weighted_mae": weighted_mae,
            "acreage_weighted_rmse": weighted_rmse,
        })

    results_table = (
        pd.DataFrame(results)
        .sort_values("rmse_bu_acre")
        .reset_index(drop=True)
    )

    return predictions_oos, results_table

In [ ]:
# ============================================================
# 6. Run all feature windows
# ============================================================

all_window_predictions = []
all_window_results = []

for feature_window, core_features in feature_sets.items():
    print(f"\nRunning {feature_window} with {len(core_features)} features")

    preds, res = run_rolling_oos(
        model_data=model_data,
        core_features=core_features,
        feature_window=feature_window,
        test_years=range(2018, 2026)
    )

    all_window_predictions.append(preds)
    all_window_results.append(res)

predictions_all_windows = pd.concat(
    all_window_predictions,
    ignore_index=True
)

window_results_table = pd.concat(
    all_window_results,
    ignore_index=True
)

window_order = {
    "July-end": 1,
    "August-end": 2,
    "September-end": 3,
}

window_results_table["window_order"] = (
    window_results_table["feature_window"].map(window_order)
)

window_results_table = (
    window_results_table
    .sort_values(["window_order", "rmse_bu_acre"])
    .drop(columns="window_order")
    .reset_index(drop=True)
)

display(window_results_table.round(4))

## 4. Model performance across feature windows

This table is the main model-performance result. It compares Linear Regression, Random Forest, XGBoost, and a historical-mean benchmark across July-end, August-end, and September-end feature windows. The key expectation is that RMSE should decline and OOS R² should improve as more growing-season information becomes available.


In [ ]:
# ============================================================
# 7. Save model performance tables
# ============================================================

window_results_table.to_csv(
    OUTPUT_TABLE_DIR / "model_performance_by_feature_window.csv",
    index=False
)

best_model_by_window = (
    window_results_table
    .sort_values(["feature_window", "rmse_bu_acre"])
    .groupby("feature_window", as_index=False)
    .first()
)

window_order = {
    "July-end": 1,
    "August-end": 2,
    "September-end": 3
}

best_model_by_window["window_order"] = (
    best_model_by_window["feature_window"].map(window_order)
)

best_model_by_window = (
    best_model_by_window
    .sort_values("window_order")
    .drop(columns="window_order")
    .reset_index(drop=True)
)

best_model_by_window.to_csv(
    OUTPUT_TABLE_DIR / "best_model_by_feature_window.csv",
    index=False
)

print("Saved:")
print(OUTPUT_TABLE_DIR / "model_performance_by_feature_window.csv")
print(OUTPUT_TABLE_DIR / "best_model_by_feature_window.csv")

display(best_model_by_window.round(4))


## 5. Aggregate county-level predictions into satellite-implied yield

The county-level XGBoost predictions are aggregated into a top-15-state satellite-implied yield. The aggregation uses prior-year county harvested acres from the USDA county outcome file as weights.

This produces one aggregate satellite yield estimate per feature window and crop year.


In [ ]:
# ============================================================
# Build top-15 satellite-implied aggregate yield
# weighted by lagged county harvested acres
# for July-end, August-end, September-end
# ============================================================

import pandas as pd
import numpy as np

signal_model = "XGBoost"

# ------------------------------------------------------------
# 1. Standardize keys in all-window predictions
# ------------------------------------------------------------

pred = predictions_all_windows[
    predictions_all_windows["model"] == signal_model
].copy()

pred["state_alpha"] = (
    pred["state_alpha"]
    .astype("string")
    .str.strip()
    .str.upper()
)

pred["county_ansi"] = (
    pd.to_numeric(pred["county_ansi"], errors="coerce")
    .astype("Int64")
    .astype("string")
    .str.zfill(3)
)

pred["year"] = (
    pd.to_numeric(pred["year"], errors="coerce")
    .astype("Int64")
)

# Check feature windows
print("Prediction rows:", len(pred))
print("Feature windows:", pred["feature_window"].unique())
print("Years:", pred["year"].min(), "-", pred["year"].max())

# ------------------------------------------------------------
# 2. Prepare lagged county harvested acres from county_outcomes
# ------------------------------------------------------------

acres_history = county_outcomes.copy()

acres_history["state_alpha"] = (
    acres_history["state_alpha"]
    .astype("string")
    .str.strip()
    .str.upper()
)

acres_history["county_ansi"] = (
    pd.to_numeric(acres_history["county_ansi"], errors="coerce")
    .astype("Int64")
    .astype("string")
    .str.zfill(3)
)

acres_history["year"] = (
    pd.to_numeric(acres_history["year"], errors="coerce")
    .astype("Int64")
)

acres_history["harvested_acres"] = pd.to_numeric(
    acres_history["harvested_acres"],
    errors="coerce"
)

acres_history = (
    acres_history[
        ["state_alpha", "county_ansi", "year", "harvested_acres"]
    ]
    .drop_duplicates(
        subset=["state_alpha", "county_ansi", "year"]
    )
    .dropna(subset=["state_alpha", "county_ansi", "year", "harvested_acres"])
    .copy()
)

# Create t+1 year so acres from year t become lagged acres for prediction year t+1
lagged_acres = acres_history.copy()
lagged_acres["year"] = lagged_acres["year"] + 1

lagged_acres = lagged_acres.rename(
    columns={"harvested_acres": "lagged_harvested_acres"}
)

# ------------------------------------------------------------
# 3. Merge lagged acres into county-level predictions
# ------------------------------------------------------------

pred_with_weights = pred.merge(
    lagged_acres,
    on=["state_alpha", "county_ansi", "year"],
    how="left",
    validate="many_to_one"
)

print("Prediction rows after merge:", len(pred_with_weights))
print("Missing lagged acres:", pred_with_weights["lagged_harvested_acres"].isna().sum())

display(pred_with_weights.head())

# ------------------------------------------------------------
# 4. Aggregate by feature window and year
# ------------------------------------------------------------

pred_with_weights = pred_with_weights.dropna(
    subset=["prediction", "lagged_harvested_acres"]
).copy()

pred_with_weights = pred_with_weights[
    pred_with_weights["lagged_harvested_acres"] > 0
].copy()

pred_with_weights["weighted_predicted_yield"] = (
    pred_with_weights["prediction"] *
    pred_with_weights["lagged_harvested_acres"]
)

satellite_yield_raw_all = (
    pred_with_weights
    .groupby(["feature_window", "model", "year"], as_index=False)
    .agg(
        weighted_predicted_yield_sum=("weighted_predicted_yield", "sum"),
        lagged_harvested_acres_sum=("lagged_harvested_acres", "sum"),
        counties_used=("county_ansi", "count")
    )
)

satellite_yield_raw_all["satellite_yield"] = (
    satellite_yield_raw_all["weighted_predicted_yield_sum"] /
    satellite_yield_raw_all["lagged_harvested_acres_sum"]
)

satellite_yield_raw_all = satellite_yield_raw_all[
    [
        "feature_window",
        "model",
        "year",
        "satellite_yield",
        "lagged_harvested_acres_sum",
        "counties_used"
    ]
].copy()

window_order = {
    "July-end": 1,
    "August-end": 2,
    "September-end": 3
}

satellite_yield_raw_all["window_order"] = (
    satellite_yield_raw_all["feature_window"].map(window_order)
)

satellite_yield_raw_all = (
    satellite_yield_raw_all
    .sort_values(["window_order", "year"])
    .drop(columns="window_order")
    .reset_index(drop=True)
)

display(satellite_yield_raw_all.round(4))

## 6. Rolling bias adjustment

The raw aggregate satellite yield can have systematic prediction bias. To avoid look-ahead bias, the adjustment for year *t* uses only prior out-of-sample errors:

- 2019 uses the 2018 prediction bias.
- 2020 uses the average 2018–2019 bias.
- 2021 uses the average 2018–2020 bias.

The rolling bias is computed separately for each feature window.


In [ ]:
# ============================================================
# Build rolling bias by year for July/August/September XGBoost predictions
# ============================================================

signal_model = "XGBoost"

xgb_pred_all = predictions_all_windows[
    predictions_all_windows["model"] == signal_model
].copy()

xgb_pred_all["year"] = pd.to_numeric(
    xgb_pred_all["year"],
    errors="coerce"
).astype(int)

xgb_pred_all["yield_bu_acre"] = pd.to_numeric(
    xgb_pred_all["yield_bu_acre"],
    errors="coerce"
)

xgb_pred_all["prediction"] = pd.to_numeric(
    xgb_pred_all["prediction"],
    errors="coerce"
)

# Prediction bias = predicted county yield - actual county yield
xgb_pred_all["pred_minus_actual"] = (
    xgb_pred_all["prediction"] -
    xgb_pred_all["yield_bu_acre"]
)

bias_by_year_all = (
    xgb_pred_all
    .groupby(["feature_window", "year"], as_index=False)
    .agg(
        actual_yield_mean=("yield_bu_acre", "mean"),
        pred_yield_mean=("prediction", "mean"),
        pred_minus_actual=("pred_minus_actual", "mean"),
        mae=("pred_minus_actual", lambda x: np.mean(np.abs(x)))
    )
)

# Keep windows in logical order
window_order = {
    "July-end": 1,
    "August-end": 2,
    "September-end": 3
}

bias_by_year_all["window_order"] = (
    bias_by_year_all["feature_window"].map(window_order)
)

bias_by_year_all = (
    bias_by_year_all
    .sort_values(["window_order", "year"])
    .reset_index(drop=True)
)

# Rolling bias must be computed separately within each feature window.
# 2019 uses 2018 bias.
# 2020 uses average 2018-2019 bias.
# 2021 uses average 2018-2020 bias.
bias_by_year_all["rolling_bias"] = (
    bias_by_year_all
    .groupby("feature_window")["pred_minus_actual"]
    .transform(lambda s: s.expanding().mean().shift(1))
)

bias_by_year_all = (
    bias_by_year_all
    .drop(columns="window_order")
    .reset_index(drop=True)
)

display(bias_by_year_all.round(4))

In [ ]:
# ============================================================
# Apply rolling bias adjustment to satellite aggregate yield
# ============================================================

satellite_yield_adj_all = satellite_yield_raw_all.merge(
    bias_by_year_all[
        [
            "feature_window",
            "year",
            "rolling_bias"
        ]
    ],
    on=["feature_window", "year"],
    how="left"
)

satellite_yield_adj_all["satellite_yield_adj"] = (
    satellite_yield_adj_all["satellite_yield"] -
    satellite_yield_adj_all["rolling_bias"]
)

# 2018 has no prior OOS bias, so adjusted signal starts from 2019
satellite_yield_adj_eval = (
    satellite_yield_adj_all
    .dropna(subset=["rolling_bias"])
    .copy()
)

satellite_yield_adj_eval["window_order"] = (
    satellite_yield_adj_eval["feature_window"].map(window_order)
)

satellite_yield_adj_eval = (
    satellite_yield_adj_eval
    .sort_values(["window_order", "year"])
    .drop(columns="window_order")
    .reset_index(drop=True)
)

display(satellite_yield_adj_eval.round(4))

## 7. Save July/August/September satellite yield signal

The pivot table stores July-end, August-end, and September-end satellite-implied yield signals in one row per crop year. The next notebook can read this file and select July/August columns for the WASDE bracketing test.


In [ ]:
# ============================================================
# Pivot July/August/September adjusted satellite yield by year
# ============================================================

signal_pivot = (
    satellite_yield_adj_eval
    .pivot_table(
        index="year",
        columns="feature_window",
        values=[
            "satellite_yield",
            "rolling_bias",
            "satellite_yield_adj"
        ],
        aggfunc="first"
    )
)

# Flatten multi-level columns
signal_pivot.columns = [
    f"{window}_{metric}"
    for metric, window in signal_pivot.columns
]

signal_pivot = signal_pivot.reset_index()

# Rename columns to cleaner names
signal_pivot = signal_pivot.rename(columns={
    "July-end_satellite_yield": "july_satellite_yield_raw",
    "August-end_satellite_yield": "aug_satellite_yield_raw",
    "September-end_satellite_yield": "sep_satellite_yield_raw",

    "July-end_rolling_bias": "july_rolling_bias",
    "August-end_rolling_bias": "aug_rolling_bias",
    "September-end_rolling_bias": "sep_rolling_bias",

    "July-end_satellite_yield_adj": "july_satellite_yield_adj",
    "August-end_satellite_yield_adj": "aug_satellite_yield_adj",
    "September-end_satellite_yield_adj": "sep_satellite_yield_adj",
})

# Reorder columns
cols = [
    "year",
    "july_satellite_yield_raw",
    "aug_satellite_yield_raw",
    "sep_satellite_yield_raw",
    "july_rolling_bias",
    "aug_rolling_bias",
    "sep_rolling_bias",
    "july_satellite_yield_adj",
    "aug_satellite_yield_adj",
    "sep_satellite_yield_adj",
]

signal_pivot = signal_pivot[cols]

display(signal_pivot.round(4))

In [ ]:
# ============================================================
# 11. Save satellite signal pivot for WASDE analysis
# ============================================================

signal_pivot.to_csv(
    OUTPUT_TABLE_DIR / "signal_pivot_july_aug_sept.csv",
    index=False
)

# Optional long-format table for debugging/reproducibility
satellite_yield_adj_eval.to_csv(
    OUTPUT_TABLE_DIR / "top15_satellite_yield_adjusted_all_feature_windows_long.csv",
    index=False
)

satellite_yield_raw_all.to_csv(
    OUTPUT_TABLE_DIR / "top15_satellite_yield_raw_all_feature_windows_long.csv",
    index=False
)

print("Saved:")
print(OUTPUT_TABLE_DIR / "signal_pivot_july_aug_sept.csv")
print(OUTPUT_TABLE_DIR / "top15_satellite_yield_adjusted_all_feature_windows_long.csv")
print(OUTPUT_TABLE_DIR / "top15_satellite_yield_raw_all_feature_windows_long.csv")
